### This notebook generates training data for stardist

Input the manually corrected pickle file to generate masks that will be used to train stardist.

In [ ]:
from __future__ import print_function, unicode_literals, absolute_import, division
import numpy as np
import matplotlib
matplotlib.rcParams["image.interpolation"] = 'none'
import matplotlib.pyplot as plt
from skimage.draw import disk
import pandas as pd
import cv2
import os
from tifffile import imwrite
from stardist import random_label_cmap

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

np.random.seed(42)
lbl_cmap = random_label_cmap()

################## Directory settings ####################
ROOT_DIR = os.path.join(os.getcwd(),'stardist_data', 'TPE_disk')
IMAGE_DIR = os.path.join(ROOT_DIR,'train', "images")
MASK_DIR = os.path.join(ROOT_DIR,'train', "masks")

############### Masking functions (Don't alter)####################

def generate_instance_mask(image_shape, positions, radii):
    mask = np.zeros(image_shape, dtype=np.uint16)
    for i, (pos, r) in enumerate(zip(positions, radii), 1):
        radius = r[0]
        rr, cc = disk((int(pos[1]), int(pos[0])), radius, shape=image_shape)
        mask[rr, cc] = i
    return mask


camera alignment

adjust according to latest version

In [ ]:

################ Camera alignment ####################
#may vary based on camera and setup

def camera_align(I): #updated 20250918
    H = np.array([[ 9.66562602e-01,  4.69927849e-03,  3.25853001e+01],
       [-2.45739411e-03,  9.68118454e-01,  4.89026546e+00],
       [ 9.37132228e-08, -2.21548575e-07,  1.00000000e+00]])
    height, width = I.shape[:2]
    warped_I = cv2.warpPerspective(I, H, (width, height))
    return warped_I


roi = (250,1200, 100, 1800) #ROI FOR in y and x direction #TRIM OFF TWO ENDS IN X FOR TRAINING

1. First set save = 0, plot = 1: Inspect the colored masks that corresponds to disk locations

2. If correct, set save = 1 and save to database

Corresponding frames and masks are named after frame number. To avoid overwriting when using frames from multiple experiments, add suffixes as needed.

In [ ]:
save = 0
plot = 1

In [ ]:
################ Input filenames ####################
pickle_path = r"O:\LJJ202107\TPE_track_files\TPE_20250919A01_N=280x2_L=835_W=149_VF=0.82_e-4rps_h=0.38V_-7mm.pkl"
F = pd.read_pickle(pickle_path)
circle_img_dir = r"N:\PROJ_TPE\TPE_20250919A01_N=280x2_L=835_W=149_VF=0.82_e-4rps_h=0.38V_-7mm\Ic_"

suffix = "_dense0919A01.tif" 

####################################################

for frame in [150]:
#for frame in np.arange(60,180,5):
    f = F[F.frame==frame]
    f.loc[:,'x'] = f.loc[:,'x'] - 100  #trim off left edge for training
    circle_img_path = circle_img_dir + str(frame)+ '.png'
    I_circle =  cv2.flip(cv2.imread(circle_img_path), 1) # Takes the green channel
    I_circle = camera_align(I_circle)[roi[0]:roi[1], roi[2]:roi[3], 1] #read image
    # turn to color
    I_circle = cv2.cvtColor(I_circle, cv2.COLOR_GRAY2RGB)

    size = I_circle.shape[:2]

    mask = generate_instance_mask(size,(f[['x', 'y']].values).tolist(),(f[['rpx']].values).tolist())

    if save:
        imwrite(os.path.join(IMAGE_DIR, str(frame)+suffix), I_circle)
        imwrite(os.path.join(MASK_DIR, str(frame)+suffix), mask)

if plot:

    plt.figure()
    plt.imshow(I_circle, cmap='gray')  # Remove cmap or change it if I_circle is an RGB image

    # Create a masked array so that only nonzero values are shown
    masked_overlay = np.ma.masked_where(mask == 0, mask)

    plt.imshow(masked_overlay, cmap='jet', alpha=0.4)


FileNotFoundError: [Errno 2] No such file or directory: 'O:\\LJJ202107\\TPE_track_files\\TPE_20250919A01_N=280x2_L=835_W=149_VF=0.82_e-4rps_h=0.38V_-7mm.pkl'